In [1]:
from facenet_pytorch import InceptionResnetV1

/home/user/miniconda/envs/minus_face/lib/python3.7/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
teacher = InceptionResnetV1(pretrained='vggface2').eval().to("cpu")

In [6]:
from torchvision import transforms
tf_teacher = transforms.Compose([
    transforms.Resize((160,160)),
    transforms.ToTensor(),
])

In [15]:
import pandas as pd
import random
import os
df = pd.read_csv('/path/to/Identity_CelebA.txt', sep=' ')
df.columns = ["col0","img_name","id"]

def get_another_image(df, img_name):
    row = df[df["img_name"] == img_name]
    if row.empty:
        return None
    id_val = row["id"].iloc[0]
    imgs = df[df["id"] == id_val]["img_name"].tolist()
    imgs = [x for x in imgs if x != img_name]
    if not imgs:
        return None
    return random.choice(imgs)

def select_teacher_image(path):
    if random.random() < 1:
        img_name = os.path.basename(path)
        alt = get_another_image(df, img_name)
        if alt is not None:
            return os.path.join(os.path.dirname(path), alt)
    return path

In [ ]:
from PIL import Image
images = os.listdir("training")
image1 = Image.open(os.path.join("training", images[0]))
teacher_image_path = select_teacher_image(os.path.join("training", images[0]))
image2 = Image.open(teacher_image_path) 
ft1 = teacher(tf_teacher(image1).unsqueeze(0))
ft2 = teacher(tf_teacher(image2).unsqueeze(0))
ft1 @ ft2.T

tensor([[0.6199]], grad_fn=<MmBackward0>)